In [1]:
import os

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

load_dotenv()

OPENAI_MODEL = "gpt-4o-mini"
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")


/Users/guane/Documentos/GlogalLogic/AI-course/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
try:
    # Intentar OpenAI primero
    llm = ChatOpenAI(
        model=OPENAI_MODEL,
        temperature=0,
        api_key=os.getenv("OPENAI_API_KEY"),
    )
    _ = llm.invoke("ping")
    provider = "OpenAI"
except Exception:
    # Si OpenAI falla, usar Ollama
    llm = ChatOllama(
        model=OLLAMA_MODEL,
        temperature=0,
    )
    _ = llm.invoke("ping")
    provider = "Ollama"

print(f"Proveedor activo: {provider}")
response = llm.invoke("Hola, soy Eder, tengo 25 años y me gusta el fútbol.")
response.text

Proveedor activo: OpenAI


'¡Hola, Eder! Es genial que te guste el fútbol. ¿Tienes un equipo favorito o juegas tú mismo?'

In [ ]:
class ContactInfo(BaseModel):
    """Información de contacto de una persona"""
    name: str = Field(description="El nombre de la persona")
    age: int = Field(description="La edad en años")
    occupation: str = Field(description="Profesión u ocupación", default="No especificado")
    interests: str = Field(description="Intereses o hobbies", default="No especificado")
    tone: int = Field(description="Tono del mensaje (0=negativo, 100=positivo)", ge=0, le=100)

llm_structured_output = llm.with_structured_output(schema=ContactInfo)

messages = [
    ("system", "Extrae la info: Juan tiene 30 años y es ingeniero"),
    ("user", "Hola, soy Eder, tengo 25 años y me gusta el fútbol")
]



In [12]:
messages = [
    ("system", "Eres un asistente que extrae información de contacto de los mensajes. "
               "Si no encuentras algún dato, usa valores por defecto razonables."),
    ("user", "Hola, soy Eder, tengo 25 años y me gusta el fútbol")
]

response = llm_structured_output.invoke(messages)


In [11]:
response.model_dump()

{'name': 'Eder',
 'age': 25,
 'occupation': 'Desconocida',
 'interests': 'Fútbol',
 'tone': 20}